# LeNet-5 Model

In [1]:
import torch
from torch import nn

class LeNet5(nn.Module):
    def __init__(self):
        super().__init__()
        self.feature = nn.Sequential(
            nn.Conv2d(1, 6, kernel_size=5, stride=1, padding=2), # 28 > 32
            nn.Tanh(),
            nn.AvgPool2d(kernel_size=2, stride=2),

            nn.Conv2d(6, 16, kernel_size=5, stride=1),
            nn.Tanh(),
            nn.AvgPool2d(kernel_size=2, stride=2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16*5*5, 120),
            nn.Tanh(),
            nn.Linear(120, 84),
            nn.Tanh(),
            nn.Linear(84, 10)
        )
    
    def forward(self, x: torch.Tensor):
        # we expect this input to be a flattened mnist tensor with batch dimension
        x = x.view(x.size(0), 1, 28, 28)
        x = self.feature(x)
        x = self.classifier(x)
        return x

In [2]:
import mnist

BATCH_SIZE = 12
training_loader, test_loader = mnist.get_loaders(BATCH_SIZE)

In [3]:
# initialize model
model = LeNet5()
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# train the model
import trainer

EPOCHS = 10
STOP_LOSS = 0.005
trainer.train_model(
    model,
    training_loader,
    test_loader,
    loss_fn,
    optimizer,
    print_freq=600,
    epochs=EPOCHS,
    stop_loss=STOP_LOSS,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

In [5]:
# save the model
torchscript_model = torch.jit.script(model)
torch.jit.save(torchscript_model, "torchscript-models/lenet-5.pt")